# RADAR AI - Inference Pipeline
**Track A: The Edge Vision**

Script ini didesain khusus agar dapat dijalankan sepenuhnya **offline** pada mesin bersumber daya rendah (Edge Computing). 
Sesuai dengan constraint Stage 2, proses inferensi berikut murni menggunakan **CPU** tanpa memanggil API eksternal apa pun, dengan target waktu inferensi < 3 detik dan rentang ukuran model < 50 MB, mendukung implementasi riil pada perangkat lapangan (UAV/IoT).


In [1]:
import os
import time
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
import torchvision.models as models

# --- Konfigurasi Sistem ---
# Memastikan model berjalan secara eksklusif menggunakan CPU untuk menguji
# kinerja model dalam kondisi perangkat tanpa akselerator GPU (sesuai constraint C-A2)
DEVICE = torch.device('cpu') 

MODEL_PATH = './models/model_damage_classifier.pt'
CLASS_NAMES = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']

def load_radar_model(path: str):
    """
    Memuat model MobileNetV3-Small yang dirancang khusus untuk inferensi di edge device.
    """
    # Parameter weights=None memastikan sistem tidak mencoba mengunduh pre-trained
    # weights dari internet (Offline Total, constraint C-A5).
    model = models.mobilenet_v3_small(weights=None)
    
    # Penyesuaian layer klasifikasi untuk mendeteksi 4 level kerusakan
    model.classifier[3] = nn.Linear(model.classifier[3].in_features, 4)
    
    if os.path.exists(path):
        # Memuat state_dict lokal secara eksplisit ke CPU
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        print(f'[✓] Model berhasil dimuat dari lokasi: {path}')
    else:
        print('[!] Model lokal tidak ditemukan. Pastikan proses training sudah diselesaikan terlebih dahulu.')
    
    # Mode inferensi aktif (batchnorm/dropout statis)
    return model.eval().to(DEVICE)

# --- Inisialisasi Environment ---
print('=== Inisialisasi RADAR Inference Engine ===')
print(f'Mereservasi resource hardware: {DEVICE}')
model = load_radar_model(MODEL_PATH)


def predict(img_path: str):
    """
    Fungsi prediksi utama. Menganalisis gambar individual secara on-device.
    """
    if not os.path.exists(img_path):
        print(f"Error: Gambar '{img_path}' tidak ditemukan.")
        return

    # Normalisasi standar agar distribusi data sesuai dengan yang dipelajari selama fine-tuning
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Pemrosesan gambar
    try:
        img = Image.open(img_path).convert('RGB')
        input_tensor = transform(img).unsqueeze(0).to(DEVICE)
    except Exception as e:
        print(f"Error dalam memproses gambar {img_path}: {e}")
        return
        
    # --- Blok Inferensi (Pengukuran Waktu) ---
    # t0 & t1 disematkan berdekatan dengan torch(tensor) untuk mengukur latensi CPU murni
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model(input_tensor)
    t1 = time.perf_counter()
    # -----------------------------------------
    
    prediction_index = out.argmax(1).item()
    inference_time_ms = (t1 - t0) * 1000
    
    print(f'Tingkat Kerusakan : {CLASS_NAMES[prediction_index]}')
    print(f'Kecepatan Inferensi (CPU): {inference_time_ms:.1f} milidetik')
    
    # Validasi Constraint Kecepatan (Waktu maksimum yang diizinkan adalah 3000ms / 3 detik)
    if inference_time_ms <= 3000:
         print(f'[LULUS] Waktu konputasi model memenuhi target < 3 detik.')
    else:
         print(f'[GAGAL] Mengalami delay pada saat komputasi.')

print('Engine siap memproses data.')


=== Inisialisasi RADAR Inference Engine ===
Mereservasi resource hardware: cpu
[✓] Model berhasil dimuat dari lokasi: ./models/model_damage_classifier.pt
Engine siap memproses data.


In [2]:
# --- Proses Evaluasi (Batch Test) ---
# Menguji performa engine terhadap data blind-test yang diambil dari direktori pengujian.
import glob
import os

test_directory = './test_data/test/'
sample_test_images = []

print("Memulai evaluasi sampel data acak...")

# Menarik masing-masing 1 representasi sampel dari seluruh kelas tingkat kerusakan 
# dalam dataset test dengan penanganan folder dinamis.
if os.path.exists(test_directory):
    for category in CLASS_NAMES:
        category_dir = os.path.join(test_directory, category)
        if os.path.exists(category_dir):
            file_candidates = glob.glob(os.path.join(category_dir, '*.jpg'))
            if file_candidates:
                sample_test_images.append(file_candidates[0])

if sample_test_images:
    total_images = len(sample_test_images)
    print(f"\nMenemukan {total_images} gambar sampel untuk uji performa CPU.")
    print("-" * 50)
    for idx, path in enumerate(sample_test_images, 1):
        filename = os.path.basename(path)
        print(f"[{idx}/{total_images}] Memproses: {filename}")
        predict(path)
        print("-" * 50)
else:
    print(f"[!] Gagal menemukan data testing di {test_directory}. Mohon letakkan gambar test pada struktur folder yang sesuai.")


Memulai evaluasi sampel data acak...

Menemukan 4 gambar sampel untuk uji performa CPU.
--------------------------------------------------
[1/4] Memproses: dinas-pupr-riau-perbaiki-infrastruk_aug14.jpg
Tingkat Kerusakan : no-damage
Kecepatan Inferensi (CPU): 15.8 milidetik
[LULUS] Waktu konputasi model memenuhi target < 3 detik.
--------------------------------------------------
[2/4] Memproses: 032563600_1741068459-3852db7c-0664-4131-8279-40f8f39e866d_aug15.jpg
Tingkat Kerusakan : minor-damage
Kecepatan Inferensi (CPU): 8.6 milidetik
[LULUS] Waktu konputasi model memenuhi target < 3 detik.
--------------------------------------------------
[3/4] Memproses: 1e9c5df376a75d3de7e7df68fbe0c77c-IMG_20251206_WA0017_aug16.jpg
Tingkat Kerusakan : major-damage
Kecepatan Inferensi (CPU): 8.2 milidetik
[LULUS] Waktu konputasi model memenuhi target < 3 detik.
--------------------------------------------------
[4/4] Memproses: 59971-rumah-ambruk_aug14.jpg
Tingkat Kerusakan : destroyed
Kecepatan Inf